In [1]:

import kagglehub
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

path = kagglehub.dataset_download(
    "ziya07/student-mental-health-and-resilience-dataset"
)

print("Path to dataset files:", path)

files = os.listdir(path)

csv_files = [file for file in files if file.endswith(".csv")]

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV file found in the downloaded dataset.")

csv_file = csv_files[0]

print("\nCSV file:", csv_file)

df = pd.read_csv(
    os.path.join(path, csv_file),
    engine="python",
    on_bad_lines="skip"
)

print("\nFirst 5 Rows")
print(df.head())

print("\nMissing Values")
print(df.isnull().sum())

for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].mean())

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing Values After Filling")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

df.drop_duplicates(inplace=True)

print("Duplicates after removal:", df.duplicated().sum())

num_cols = df.select_dtypes(include=np.number).columns

print("\nOutlier Detection")

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower) |
        (df[col] > upper)
    ]

    print(f"{col}: {len(outliers)} outliers")

scaler = MinMaxScaler()

df[num_cols] = scaler.fit_transform(df[num_cols])

encoder = LabelEncoder()

for col in df.select_dtypes(include="object").columns:
    df[col] = encoder.fit_transform(df[col].astype(str))

print("\nCleaned Dataset")
print(df.head())

print("\nFinal Data Types")
print(df.dtypes)

print("\nFinal Dataset Shape")
print(df.shape)

100%|██████████| 24.3k/24.3k [00:00<00:00, 29.9MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/ziya07/student-mental-health-and-resilience-dataset/versions/1

CSV file: mental_health_dataset.csv

First 5 Rows
   Student_ID  Age  Gender   GPA  Stress_Level  Anxiety_Score  \
0           1   23   Other  2.52             5             20   
1           2   19    Male  2.74             5              3   
2           3   21  Female  3.53             5             11   
3           4   18    Male  2.04             4             15   
4           5   19   Other  2.87             1              2   

   Depression_Score                                  Daily_Reflections  \
0                 6  Onto foreign do environmental anyone every nea...   
1                 7  Party but others visit admit industry country ...   
2                24  Religious sure wait do chance decade according...   
3                14            A task effect entire coach join series.   
4                 4  Knowledge several camera wait